In [ ]:
import pandas as pd 
import numpy as np
import sys
sys.path.append("..")        
from src.validation.control_datos import validar_warehouse
import pandera.errors as pe   # arriba, con los demás imports

In [ ]:
df = pd.read_parquet(f'../data/interim/tabla_maestra_estructural.parquet')

In [ ]:

cols_aemet = df.select_dtypes('object').columns

# reemplazo de centinelas ANTES de tocar comas y tipos
df[cols_aemet] = df[cols_aemet].replace({'Ip': 0, 'Acum': np.nan})

In [ ]:
(df[cols_aemet] == 'Ip').sum().sum()

In [ ]:

for column in cols_aemet: 
 df[column] =  df[column].str.replace(',', '.').astype(float)
 



In [ ]:
df[cols_aemet].dtypes

In [ ]:
# --- Verificación: ¿los precio_espana == 0 son reales o huecos enmascarados? ---
ceros = df[df['precio_espana'] == 0.0].copy()

# 1) ¿Existen precios negativos? Si el mercado va a negativo, el 0 no es un centinela
n_negativos = (df['precio_espana'] < 0).sum()
print(f"Precios negativos: {n_negativos} | mínimo: {df['precio_espana'].min():.2f} €/MWh")

# 2) Distribución horaria de los ceros (esperamos pico a mediodía solar)
ceros['hora_utc'] = ceros['datetime_utc'].dt.hour
print("\nCeros por hora UTC:")
print(ceros['hora_utc'].value_counts().sort_index())

# 3) Generación renovable en las filas con precio 0 vs. el resto
print("\nSolar FV real (mediana) en filas con precio 0:",
      ceros['solar_fv_real'].median())
print("Solar FV real (mediana) en el resto:",
      df.loc[df['precio_espana'] > 0, 'solar_fv_real'].median())

# 4) Tendencia por año (esperamos crecimiento por + capacidad solar instalada)
ceros['ano'] = ceros['datetime_utc'].dt.year
print("\nCeros por año:")
print(ceros['ano'].value_counts().sort_index())

In [ ]:
mask_cero_mediodia = df['datetime_utc'].dt.hour.between(10,14) & (df['precio_espana']==0)
mask_positivo_mediodia = df['datetime_utc'].dt.hour.between(10,14) & (df['precio_espana'] > 0 )

In [ ]:
print(df.loc[mask_cero_mediodia, 'solar_fv_real'].median())
print(df.loc[mask_positivo_mediodia, 'solar_fv_real'].median())

### Ceros y negativos en `precio_espana`: reales, se conservan

Se verificó que los 931 valores de `precio_espana == 0` no son huecos enmascarados, sino precio marginal cero por exceso de generación renovable. Evidencia:

1. **Existen 890 precios negativos** (mínimo −9,82 €/MWh). Un valor centinela nunca produciría una distribución continua que cruza el cero hacia abajo; el 0 es un punto más de esa cola.
2. **Los ceros se concentran en las horas 10–14 UTC**, coincidiendo con el pico de generación solar fotovoltaica.
3. **Control por hora del día:** restringiendo la comparación a la misma franja de mediodía (10–14 UTC) para eliminar el efecto de las horas nocturnas, la mediana de `solar_fv_real` sigue siendo mayor en las horas de precio cero (175.692 MWh) que en las de precio positivo (160.517 MWh). La diferencia (~9%) confirma la relación una vez descontado el sesgo horario.

El menor tamaño de la diferencia a igual hora indica que el colapso del precio no depende de la solar de forma aislada, sino del **balance neto** (renovable total frente a demanda) — un comportamiento que el modelo debe capturar a partir de la interacción de varias variables.

**Decisión:** se conservan ceros y negativos por ser señal de mercado relevante para el modelo.

In [ ]:
try:
    df = validar_warehouse(df)
    print("Validación OK →", df.shape)
    df.to_parquet("../data/processed/tabla_maestra_procesada.parquet")
except pe.SchemaErrors as e:
    print("La validación falló. Incumplimientos:")
    print(e.failure_cases[["column", "check", "failure_case"]].drop_duplicates())                                   